In [11]:
import torch
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms as T
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import torchmetrics

import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
from PIL import Image

In [12]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, transform = None, is_test = False):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.is_test = is_test

        if not self.is_test:
            self.classes = sorted(self.df['type'].unique())
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
            self.idx_to_class = {i: cls_name for i, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id']
        image = Image.open(img_id).convert('RGB')
        w, h = image.size
        cw, ch = w // 4, h // 4

        if self.is_test:
            cells = []
            for r in range(4):
                for c in range(4):
                    patch = image.crop((c * cw, r * ch, (c + 1) * cw, (r + 1) * ch))
                    if self.transform:
                        patch = self.transform(patch)
                    cells.append(patch)
            return torch.stack(cells), img_id
        else:
            patch = image.crop((0, 0, cw, ch))
            if self.transform:
                patch = self.transform(patch)
            label = self.class_to_idx[row['type']]
            return patch, torch.tensor(label, dtype=torch.long)

In [13]:
class DataModule(L.LightningDataModule):
    def __init__(
        self, train_csv, test_csv, batch_size=32, img_size=224, val_split=0.2
    ):
        super().__init__()
        self.train_csv = train_csv
        self.test_csv = test_csv
        self.batch_size = batch_size
        self.img_size = img_size
        self.val_split = val_split

        self.transform_train = T.Compose([
            T.Grayscale(num_output_channels=3),
            T.Resize((img_size,img_size)),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.92, 1.08)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

        self.transform_val = T.Compose([
            T.Grayscale(num_output_channels=3),
            T.Resize((img_size,img_size)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

    def setup(self, stage=None):
        full_dataset = ImageDataset(self.train_csv, transform=self.transform_train)
        self.num_classes = len(full_dataset.classes)
        self.class_to_idx = full_dataset.class_to_idx
        self.idx_to_class = full_dataset.idx_to_class

        train_idx, val_idx = train_test_split(
            range(len(full_dataset)),
            test_size=self.val_split,
            random_state=42,
            stratify=full_dataset.df['type']
        )

        self.train_dataset = Subset(full_dataset, train_idx)

        val_dataset_eval = ImageDataset(self.train_csv, transform=self.transform_val)
        self.val_dataset = Subset(val_dataset_eval, val_idx)

        self.test_dataset = ImageDataset(self.test_csv, transform=self.transform_val, is_test=True)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            persistent_workers=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

In [14]:
class ClassifierModule(L.LightningModule):
    def __init__(self, lr=3e-4):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)

        self.criterion = nn.BCEWithLogitsLoss()

        self.train_acc = torchmetrics.Accuracy(task='binary')
        self.val_acc = torchmetrics.Accuracy(task='binary')

    def forward(self, x):
        return self.backbone(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        self.train_acc(preds, y.long())

        self.log('train_loss', loss, on_epoch=True, on_step=True, prog_bar=True, logger=True)
        self.log('train_acc', self.train_acc, on_epoch=True, on_step=True, prog_bar=True, logger=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        self.val_acc(preds, y.long())

        self.log('val_loss', loss, on_epoch=True, on_step=False, prog_bar=True, logger=True)
        self.log('val_acc', self.val_acc, on_epoch=True, on_step=False, prog_bar=True, logger=True)


    def configure_optimizers(self):
        opt = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        sch = optim.lr_scheduler.ReduceLROnPlateau(optimizer=opt, mode='min', factor=0.1, patience=2)

        return {
            'optimizer':opt,
            'lr_scheduler' :{
                'scheduler': sch,
                'monitor': 'val_loss'
            }
        }

In [15]:
dm = DataModule(
    train_csv='train.csv', test_csv='test.csv', batch_size=32, img_size=224
)
dm.setup()

model = ClassifierModule(lr=3e-4)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints/',
    filename='arbin-resnet-{epoch:02d}-{val_loss:.4f}',
    save_top_k=1,
    mode='min'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss', patience=5, mode='min'
)

trainer = L.Trainer(
    max_epochs=20,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    precision='16-mixed'
)

trainer.fit(model, dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | backbone  | ResNet            | 11.2 M | train | 0    
1 | criterion | BCEWithLogitsLoss | 0      | train | 0    
2 | train_acc | BinaryAccuracy    | 0      | train | 0    
3 | val_acc   | BinaryAccuracy    | 0    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 6: 100%|██████████| 10/10 [00:00<00:00, 12.95it/s, v_num=1, train_loss_step=5.9e-5, train_acc_step=1.000, val_loss=3.7e-5, val_acc=1.000, train_loss_epoch=0.000167, train_acc_epoch=1.000]   


In [17]:
best_model_path = checkpoint_callback.best_model_path
model = ClassifierModule.load_from_checkpoint(best_model_path)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

test_loader = dm.test_dataloader()
results = []

with torch.no_grad():
    for images, img_ids in test_loader:
        B, num_classes, C, H, W = images.shape
        images_flat = images.view(B * num_classes, C, H, W).to(device)

        outputs = model(images_flat)
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        preds_grid = preds.reshape(B, 16)

        for img_id, bits in zip(img_ids, preds_grid):
            subtask1_ans = int(bits[0])
            bit_str = "".join(map(str, bits))
            subtask2_ans = int(bit_str, 2)

            results.append({
                'subtaskID': 1,
                'ID': img_id,
                'answer': subtask1_ans
            })
            results.append({
                'subtaskID': 2,
                'ID': img_id,
                'answer': subtask2_ans
            })

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)